# FakeNewsDetection – teljes Kaggle-pipeline

**Beállítások** (jobb oldali panel): Accelerator **GPU T4 x2** vagy **P100**, Internet **On**.
**Input**: *Fakeddit Dataset* (TSV-k) + *Fakeddit-50k* (képek). Később: a saját mentett embedding-datasetet is.

Sorrend: 0. adat → 1. embeddingek → 2. mag (M1–M6) → 3. extrák (M7–M8, ablációk) → 4. szignifikancia, SHAP, ábrák → 5. export a webappnak.
Előtte pushold a repót GitHubra, mert a notebook onnan tölti le a kódot.

In [ ]:
!git clone -q https://github.com/funnymrmagyar/FakeNewsDetection.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1
!pip install -q tabulate shap

In [ ]:
import glob, os
DATA, OUT = '/kaggle/working/data', '/kaggle/working/results'
# Ha egy korábbi munkamenetben elmentetted az embeddingeket datasetként, azt használjuk (nem kell újra kinyerni)
saved = glob.glob('/kaggle/input/**/text_bert.npy', recursive=True)
EMB = os.path.dirname(saved[0]) if saved else '/kaggle/working/emb'
REUSE = bool(saved)
tsv = glob.glob('/kaggle/input/**/multimodal_train.tsv', recursive=True)
img = [d for d in glob.glob('/kaggle/input/**/fakeddit_small_img', recursive=True) if os.path.isdir(d)]
assert img, 'Hiányzik a Fakeddit-50k képmappa (kell a BLIP/CLIP kinyeréshez is)'
TSV_DIR, IMG_DIR = (os.path.dirname(tsv[0]) if tsv else None), img[0]
SPLITS = f'{EMB}/splits.csv' if REUSE else f'{DATA}/splits.csv'
print('REUSE' if REUSE else 'ÚJ FUTÁS', EMB, IMG_DIR, sep='\n')

## 0. fázis – adat, címkék, felosztás
Ellenőrizd a kiírt **címkekódolást** (r/theonion → fake) és a stats-táblát – ez a dolgozat adatstatisztika-táblája.

In [ ]:
if not REUSE:
    !python -m src.data.prepare --tsv-dir "$TSV_DIR" --img-dir "$IMG_DIR" --out $DATA --check-images
    import pandas as pd; display(pd.read_csv(f'{DATA}/stats.csv'))

## 1. fázis – embeddingek
Egyszer kell lefuttatni (~20–40 perc). Utána: **Save Version** → az output `emb` mappájából **New Dataset**, és a következő munkamenetekben add hozzá inputként.

In [ ]:
if not REUSE:
    !python -m src.data.extract --splits $SPLITS --out $EMB --which text_bert img_vit img_resnet18 --batch-size 64

## 2. fázis – a mag: M1–M6 (10 seed, modellenként néhány perc)

In [ ]:
common = f'--emb-dir {EMB} --splits {SPLITS} --out {OUT}'
!python -m src.training.train --name M1_text        --features text_bert               {common}
!python -m src.training.train --name M2_resnet      --features img_resnet18            {common}
!python -m src.training.train --name M3_vit         --features img_vit                 {common}
!python -m src.eval.late_fusion --a M1_text --b M2_resnet --name M4_late_fusion --out {OUT}
!python -m src.training.train --name M5_bert_resnet --features text_bert img_resnet18  {common}
!python -m src.training.train --name M6_bert_vit    --features text_bert img_vit       {common}

In [ ]:
!python -m src.eval.report --out $OUT
!python -m src.stats.significance --out $OUT --pairs M3_vit:M2_resnet M6_bert_vit:M1_text M6_bert_vit:M5_bert_resnet M6_bert_vit:M4_late_fusion

## 3. fázis – extrák (csak ha az M1–M6 rendben van)
A BLIP-aláírás lassú (kb. 1–2 óra 50k képre T4-en), a CLIP gyors. Ha az embedding-mappa egy input-dataset (csak olvasható), az új fájlok a `/kaggle/working/emb2`-be kerülnek, ezért átmásoljuk.

In [ ]:
import shutil
if REUSE:  # az input csak olvasható: a meglévő fájlokat átmásoljuk egy írható mappába
    shutil.copytree(EMB, '/kaggle/working/emb2', dirs_exist_ok=True)
    EMB, SPLITS = '/kaggle/working/emb2', '/kaggle/working/emb2/splits.csv'
    common = f'--emb-dir {EMB} --splits {SPLITS} --out {OUT}'
!python -m src.data.extract --splits $SPLITS --out $EMB --which caption_bert clip_score

In [ ]:
!python -m src.training.train --name M7_bert_blip_vit --features text_bert img_vit caption_bert            {common}
!python -m src.training.train --name M8_bert_blip_vit_clip --features text_bert img_vit caption_bert clip_score {common}
# abláció: az M7-ből egyenként elhagyva egy-egy ágat (−aláírás = M6)
!python -m src.training.train --name A1_no_text  --features img_vit caption_bert   {common}
!python -m src.training.train --name A2_no_image --features text_bert caption_bert {common}

## 4. Eredmények: táblák, szignifikancia, SHAP, ábrák

In [ ]:
!python -m src.eval.report --out $OUT --roc M1_text M3_vit M5_bert_resnet M6_bert_vit M7_bert_blip_vit --cm M6_bert_vit M7_bert_blip_vit
!python -m src.stats.significance --out $OUT --pairs M3_vit:M2_resnet M6_bert_vit:M1_text M6_bert_vit:M5_bert_resnet M6_bert_vit:M4_late_fusion M7_bert_blip_vit:M6_bert_vit M8_bert_blip_vit_clip:M7_bert_blip_vit
import pandas as pd
display(pd.read_csv(f'{OUT}/table_main.csv'))
display(pd.read_csv(f'{OUT}/significance.csv'))

In [ ]:
!python -m src.explain.shap_analysis --ckpt $OUT/models/M7_bert_blip_vit_seed42.pt --emb-dir $EMB --splits $SPLITS --out $OUT --per-dim
from IPython.display import Image as Img
display(Img(f'{OUT}/shap/M7_bert_blip_vit_modality.png'), Img(f'{OUT}/roc.png'))

## 5. Export a webappnak
A legjobb seed fejét `fusion_head.pt` néven menti. Töltsd le (Output panel), és tedd ide: `backend/model/fusion_head.pt`.
Ha M7-et nem futtattad, írd át `M6_bert_vit`-re. A `results` mappát zip-ben is letöltheted a dolgozathoz.

In [ ]:
import pandas as pd, shutil
MODEL = 'M7_bert_blip_vit'
s = pd.read_csv(f'{OUT}/summary.csv').drop_duplicates(['name', 'seed'], keep='last')
best = s[s.name == MODEL].sort_values('val_f1', ascending=False).iloc[0]   # val alapján, nem teszt alapján!
shutil.copy(f"{OUT}/models/{MODEL}_seed{int(best.seed)}.pt", '/kaggle/working/fusion_head.pt')
shutil.make_archive('/kaggle/working/results_export', 'zip', OUT, '.')
print(f'fusion_head.pt <- {MODEL} seed {int(best.seed)} (val F1 {best.val_f1:.4f}, test F1 {best.f1:.4f})')